# Historical Simulation Stability Analysis - 09/08/2026

This notebook reproduces the Historical Simulation stability review for exception days, VaR levels, minimal rolling-window sensitivity and canonical baseline metrics.

In [1]:
import numpy as np
import pandas as pd

from src.models.historical_var import historical_var_forecast

In [2]:
RETURNS_PATH = "data/processed/portfolio_returns.csv"
BACKTEST_PATH = "data/processed/historical_var_backtest.csv"

ALPHA = 0.05
CONFIDENCE_LEVEL = 0.95
BASELINE_WINDOW = 250
SENSITIVITY_WINDOWS = [125, 250, 500]

## Canonical backtest audit

Validate the canonical 250-observation Historical Simulation backtest before stability analysis.

In [3]:
backtest = pd.read_csv(BACKTEST_PATH, parse_dates=["window_start_date", "window_end_date", "forecast_date", "target_date"])

expected_columns = ["window_start_date", "window_end_date", "forecast_date", "target_date", "observations", "quantile_return", "historical_var", "target_return", "violation"]
assert list(backtest.columns) == expected_columns
assert len(backtest) == 1387
assert backtest.isna().sum().sum() == 0
assert backtest["target_date"].duplicated().sum() == 0
assert backtest["observations"].eq(BASELINE_WINDOW).all()

var_difference = np.abs(backtest["historical_var"].to_numpy() - np.maximum(0.0, -backtest["quantile_return"].to_numpy()))
expected_violation = backtest["target_return"] < backtest["quantile_return"]

assert var_difference.max() <= 1e-12
assert backtest["violation"].eq(expected_violation).all()

print(f"Rows: {len(backtest)}")
print(f"Target date start: {backtest['target_date'].min().date()}")
print(f"Target date end: {backtest['target_date'].max().date()}")
print(f"Violations: {int(backtest['violation'].sum())}")
print(f"Violation rate: {backtest['violation'].mean():.6%}")
print(f"Maximum VaR convention difference: {var_difference.max():.3e}")

Rows: 1387
Target date start: 2020-12-31
Target date end: 2026-07-28
Violations: 75
Violation rate: 5.407354%
Maximum VaR convention difference: 0.000e+00


## Exception-day audit

Review the 75 VaR exception days and quantify the amount by which realized returns crossed the forecast quantile.

In [4]:
exceptions = backtest.loc[backtest["violation"], ["forecast_date", "target_date", "target_return", "quantile_return", "historical_var"]].copy()
exceptions["loss_excess"] = exceptions["quantile_return"] - exceptions["target_return"]

equality_count = int((backtest["target_return"] == backtest["quantile_return"]).sum())
invalid_exception_count = int((exceptions["target_return"] >= exceptions["quantile_return"]).sum())

assert len(exceptions) == 75
assert exceptions["target_date"].is_monotonic_increasing
assert exceptions["target_date"].duplicated().sum() == 0
assert invalid_exception_count == 0
assert equality_count == 0
assert (exceptions["loss_excess"] > 0.0).all()

middle_index = len(exceptions) // 2
middle_sample = exceptions.iloc[middle_index - 2:middle_index + 3]

print(f"Exceptions: {len(exceptions)}")
print(f"Minimum loss excess: {exceptions['loss_excess'].min():.6%}")
print(f"Maximum loss excess: {exceptions['loss_excess'].max():.6%}")

print("\nFIRST 5 EXCEPTIONS")
display(exceptions.head(5))

print("\nMIDDLE 5 EXCEPTIONS")
display(middle_sample)

print("\nLAST 5 EXCEPTIONS")
display(exceptions.tail(5))

Exceptions: 75
Minimum loss excess: 0.017995%
Maximum loss excess: 5.204671%

FIRST 5 EXCEPTIONS


,forecast_date,target_date,target_return,quantile_return,historical_var,loss_excess
12,2021-01-18,2021-01-19,-0.060883,-0.037728,0.037728,0.023155
19,2021-01-27,2021-01-28,-0.069552,-0.036194,0.036194,0.033357
73,2021-04-20,2021-04-22,-0.024475,-0.024062,0.024062,0.000413
75,2021-04-23,2021-04-26,-0.026977,-0.024782,0.024782,0.002195
104,2021-06-07,2021-06-08,-0.024962,-0.024782,0.024782,0.000180



MIDDLE 5 EXCEPTIONS


,forecast_date,target_date,target_return,quantile_return,historical_var,loss_excess
685,2023-10-02,2023-10-03,-0.039877,-0.035094,0.035094,0.004783
695,2023-10-16,2023-10-17,-0.031041,-0.030237,0.030237,0.000803
702,2023-10-25,2023-10-26,-0.043928,-0.028450,0.028450,0.015478
705,2023-10-30,2023-10-31,-0.030813,-0.030237,0.030237,0.000576
722,2023-11-22,2023-11-23,-0.045536,-0.024505,0.024505,0.021031



LAST 5 EXCEPTIONS


,forecast_date,target_date,target_return,quantile_return,historical_var,loss_excess
1298,2026-03-20,2026-03-23,-0.034915,-0.027020,0.027020,0.007895
1380,2026-07-17,2026-07-20,-0.025060,-0.023372,0.023372,0.001688
1382,2026-07-21,2026-07-22,-0.025912,-0.024635,0.024635,0.001277
1384,2026-07-23,2026-07-24,-0.025561,-0.025074,0.025074,0.000488
1385,2026-07-24,2026-07-27,-0.033970,-0.025347,0.025347,0.008623


## Exception severity

Rank VaR exception days by the amount the realized return crossed below the forecast quantile. This section describes severity only and does not attribute market causes.

In [5]:
ranked_exceptions = exceptions.sort_values("loss_excess", ascending=False).reset_index(drop=True)
top10_exceptions = ranked_exceptions.head(10)

severity_mean = float(exceptions["loss_excess"].mean())
severity_median = float(exceptions["loss_excess"].median())
severity_q25 = float(exceptions["loss_excess"].quantile(0.25))
severity_q75 = float(exceptions["loss_excess"].quantile(0.75))
severity_q90 = float(exceptions["loss_excess"].quantile(0.90))

assert len(ranked_exceptions) == 75
assert (ranked_exceptions["loss_excess"] > 0.0).all()
assert ranked_exceptions["target_date"].duplicated().sum() == 0
assert top10_exceptions["loss_excess"].is_monotonic_decreasing

print(f"Mean loss excess: {severity_mean:.6%}")
print(f"Median loss excess: {severity_median:.6%}")
print(f"25th percentile: {severity_q25:.6%}")
print(f"75th percentile: {severity_q75:.6%}")
print(f"90th percentile: {severity_q90:.6%}")
print(f"Maximum loss excess: {ranked_exceptions.loc[0, 'loss_excess']:.6%}")

print("\nTOP 10 MOST SEVERE EXCEPTIONS")
display(top10_exceptions)


Mean loss excess: 1.319919%
Median loss excess: 0.789465%
25th percentile: 0.288329%
75th percentile: 1.871503%
90th percentile: 3.345445%
Maximum loss excess: 5.204671%

TOP 10 MOST SEVERE EXCEPTIONS


,forecast_date,target_date,target_return,quantile_return,historical_var,loss_excess
0,2025-04-02,2025-04-03,-0.069840,-0.017793,0.017793,0.052047
1,2025-04-04,2025-04-08,-0.069299,-0.018428,0.018428,0.050871
2,2022-04-22,2022-04-25,-0.069485,-0.021813,0.021813,0.047672
3,2026-03-06,2026-03-09,-0.069402,-0.025013,0.025013,0.044389
4,2021-07-05,2021-07-06,-0.064361,-0.023479,0.023479,0.040882
5,2022-05-06,2022-05-09,-0.058539,-0.022322,0.022322,0.036217
6,2022-09-30,2022-10-03,-0.060254,-0.024092,0.024092,0.036162
7,2022-05-11,2022-05-12,-0.055981,-0.022462,0.022462,0.033519
8,2021-01-27,2021-01-28,-0.069552,-0.036194,0.036194,0.033357
9,2022-10-20,2022-10-21,-0.061691,-0.028639,0.028639,0.033052


## VaR level review

Summarize the level and dispersion of the canonical Historical VaR forecasts and identify the minimum and maximum VaR observations.

In [6]:
var_series = backtest["historical_var"].astype("float64")

var_summary = pd.Series({
    "average_var": var_series.mean(), "median_var": var_series.median(), "std_var": var_series.std(),
    "minimum_var": var_series.min(), "q25_var": var_series.quantile(0.25),
    "q75_var": var_series.quantile(0.75), "maximum_var": var_series.max(),
})

minimum_row = backtest.loc[var_series.idxmin(), ["forecast_date", "target_date", "quantile_return", "historical_var", "target_return", "violation"]]
maximum_row = backtest.loc[var_series.idxmax(), ["forecast_date", "target_date", "quantile_return", "historical_var", "target_return", "violation"]]

assert np.isfinite(var_series).all()
assert (var_series >= 0.0).all()
assert var_summary["minimum_var"] <= var_summary["average_var"] <= var_summary["maximum_var"]

display(var_summary.to_frame("value"))
print("\nMINIMUM VAR ROW")
display(minimum_row.to_frame("value"))
print("\nMAXIMUM VAR ROW")
display(maximum_row.to_frame("value"))

,value
average_var,0.026231
median_var,0.024297
std_var,0.006929
minimum_var,0.017088
q25_var,0.022322
q75_var,0.027020
maximum_var,0.043786



MINIMUM VAR ROW


,value
forecast_date,2024-10-31 00:00:00
target_date,2024-11-01 00:00:00
quantile_return,-0.017088
historical_var,0.017088
target_return,-0.009982
violation,False



MAXIMUM VAR ROW


,value
forecast_date,2022-12-26 00:00:00
target_date,2022-12-27 00:00:00
quantile_return,-0.043786
historical_var,0.043786
target_return,0.032288
violation,False


## VaR neighborhoods around the extremes

Inspect the five forecasts before and after the minimum and maximum VaR observations. The purpose is to describe local step changes in the rolling empirical quantile.

In [7]:
def build_var_neighborhood(center_idx, radius=5):
    start = max(0, center_idx - radius)
    end = min(len(backtest), center_idx + radius + 1)
    block = backtest.loc[start:end - 1, ["forecast_date", "target_date", "historical_var", "quantile_return", "target_return", "violation"]].copy()
    block["var_change"] = block["historical_var"].diff()
    block["abs_var_change"] = block["var_change"].abs()
    assert block["forecast_date"].duplicated().sum() == 0
    assert block["target_date"].duplicated().sum() == 0
    assert block["historical_var"].ge(0.0).all()
    return block

minimum_neighborhood = build_var_neighborhood(int(var_series.idxmin()))
maximum_neighborhood = build_var_neighborhood(int(var_series.idxmax()))

print("NEIGHBORHOOD AROUND MINIMUM VAR")
display(minimum_neighborhood)
print(f"Maximum one-step absolute VaR change: {minimum_neighborhood['abs_var_change'].max():.6%}")

print("\nNEIGHBORHOOD AROUND MAXIMUM VAR")
display(maximum_neighborhood)
print(f"Maximum one-step absolute VaR change: {maximum_neighborhood['abs_var_change'].max():.6%}")

NEIGHBORHOOD AROUND MINIMUM VAR


,forecast_date,target_date,historical_var,quantile_return,target_return,violation,var_change,abs_var_change
951,2024-10-24,2024-10-25,0.019211,-0.019211,-0.000247,False,NaN,NaN
952,2024-10-25,2024-10-28,0.019211,-0.019211,0.005991,False,0.000000,0.000000
953,2024-10-28,2024-10-29,0.018866,-0.018866,0.010183,False,-0.000345,0.000345
954,2024-10-29,2024-10-30,0.018866,-0.018866,-0.001935,False,0.000000,0.000000
955,2024-10-30,2024-10-31,0.018068,-0.018068,0.001169,False,-0.000798,0.000798
956,2024-10-31,2024-11-01,0.017088,-0.017088,-0.009982,False,-0.000980,0.000980
957,2024-11-01,2024-11-04,0.017088,-0.017088,-0.012091,False,0.000000,0.000000
958,2024-11-04,2024-11-05,0.017088,-0.017088,0.004632,False,0.000000,0.000000
959,2024-11-05,2024-11-06,0.017088,-0.017088,0.010635,False,0.000000,0.000000
960,2024-11-06,2024-11-07,0.017088,-0.017088,0.000167,False,0.000000,0.000000


Maximum one-step absolute VaR change: 0.097987%

NEIGHBORHOOD AROUND MAXIMUM VAR


,forecast_date,target_date,historical_var,quantile_return,target_return,violation,var_change,abs_var_change
491,2022-12-19,2022-12-20,0.041008,-0.041008,-0.021126,False,NaN,NaN
492,2022-12-20,2022-12-21,0.041008,-0.041008,0.000055,False,0.000000,0.000000
493,2022-12-21,2022-12-22,0.041008,-0.041008,0.000753,False,0.000000,0.000000
494,2022-12-22,2022-12-23,0.041008,-0.041008,-0.011245,False,0.000000,0.000000
495,2022-12-23,2022-12-26,0.041008,-0.041008,-0.052945,True,0.000000,0.000000
496,2022-12-26,2022-12-27,0.043786,-0.043786,0.032288,False,0.002777,0.002777
497,2022-12-27,2022-12-28,0.043786,-0.043786,-0.006022,False,0.000000,0.000000
498,2022-12-28,2022-12-29,0.043786,-0.043786,-0.006732,False,0.000000,0.000000
499,2022-12-29,2022-12-30,0.043786,-0.043786,0.002994,False,0.000000,0.000000
500,2022-12-30,2023-01-03,0.043786,-0.043786,0.044667,False,0.000000,0.000000


Maximum one-step absolute VaR change: 0.277720%


## Minimal rolling-window sensitivity

Compare 125, 250 and 500 observation rolling windows on identical target dates. The sensitivity check is descriptive and is not used to tune the canonical 250-day baseline.

In [8]:
portfolio_returns = pd.read_csv(RETURNS_PATH, parse_dates=["date"])

assert portfolio_returns["date"].is_monotonic_increasing
assert portfolio_returns["date"].duplicated().sum() == 0
assert portfolio_returns["portfolio_simple_return"].isna().sum() == 0

available_target_dates = {}
for window in SENSITIVITY_WINDOWS:
    available_target_dates[window] = pd.DatetimeIndex(portfolio_returns["date"].iloc[window:])

common_dates = available_target_dates[SENSITIVITY_WINDOWS[0]]
for window in SENSITIVITY_WINDOWS[1:]:
    common_dates = common_dates.intersection(available_target_dates[window])

common_dates = common_dates.sort_values()
baseline_dates = pd.DatetimeIndex(backtest["target_date"])
missing_from_baseline = common_dates.difference(baseline_dates)

assert len(common_dates) == 1137
assert common_dates.is_monotonic_increasing
assert common_dates.duplicated().sum() == 0
assert len(missing_from_baseline) == 0

for window in SENSITIVITY_WINDOWS:
    dates = available_target_dates[window]
    print(f"Window {window}: {len(dates)} available forecasts, {dates.min().date()} to {dates.max().date()}")

print(f"Common evaluation rows: {len(common_dates)}")
print(f"Common evaluation period: {common_dates.min().date()} to {common_dates.max().date()}")

Window 125: 1512 available forecasts, 2020-07-08 to 2026-07-28
Window 250: 1387 available forecasts, 2020-12-31 to 2026-07-28
Window 500: 1137 available forecasts, 2021-12-31 to 2026-07-28
Common evaluation rows: 1137
Common evaluation period: 2021-12-31 to 2026-07-28


### Sensitivity forecasts and canonical regression

Recompute Historical Simulation forecasts for all three rolling windows on the common evaluation dates using the production model. The 250-day results are regressed against the canonical backtest to confirm numerical equivalence.

In [9]:
return_dates = pd.DatetimeIndex(portfolio_returns["date"])
return_values = portfolio_returns["portfolio_simple_return"].to_numpy(dtype="float64")
date_to_index = {date: idx for idx, date in enumerate(return_dates)}
sensitivity_results = {}

for window in SENSITIVITY_WINDOWS:
    rows = []
    for target_date in common_dates:
        target_idx = date_to_index[target_date]
        training_returns = return_values[target_idx - window:target_idx]
        forecast = historical_var_forecast(training_returns, alpha=ALPHA)
        actual_return = float(return_values[target_idx])
        rows.append({
            "forecast_date": return_dates[target_idx - 1], "target_date": target_date, "observations": len(training_returns),
            "actual_return": actual_return, "quantile_return": forecast["quantile_return"], "var": forecast["var"],
            "violation": actual_return < forecast["quantile_return"],
        })

    result = pd.DataFrame(rows)
    sign_difference = np.abs(result["var"].to_numpy() - np.maximum(0.0, -result["quantile_return"].to_numpy()))
    assert len(result) == len(common_dates) == 1137
    assert result.isna().sum().sum() == 0
    assert result["target_date"].duplicated().sum() == 0
    assert result["observations"].eq(window).all()
    assert result["var"].ge(0.0).all()
    assert sign_difference.max() <= 1e-12
    assert result["violation"].eq(result["actual_return"] < result["quantile_return"]).all()
    assert (result["forecast_date"] < result["target_date"]).all()
    sensitivity_results[window] = result

    print(f"Window {window}")
    print(f"Rows: {len(result)}")
    print(f"Observations: {result['observations'].min()} to {result['observations'].max()}")
    print(f"Violations: {int(result['violation'].sum())}")
    print(f"Minimum VaR: {result['var'].min():.6%}")
    print(f"Maximum VaR: {result['var'].max():.6%}")
    print(f"Maximum sign convention difference: {sign_difference.max():.3e}\n")

baseline_common = backtest.loc[backtest["target_date"].isin(common_dates)].sort_values("target_date").reset_index(drop=True)
result_250 = sensitivity_results[BASELINE_WINDOW].sort_values("target_date").reset_index(drop=True)

assert len(baseline_common) == len(result_250) == 1137
assert baseline_common["target_date"].equals(result_250["target_date"])

quantile_difference = np.abs(result_250["quantile_return"].to_numpy() - baseline_common["quantile_return"].to_numpy())
var_difference_250 = np.abs(result_250["var"].to_numpy() - baseline_common["historical_var"].to_numpy())

assert quantile_difference.max() <= 1e-12
assert var_difference_250.max() <= 1e-12

print("250-DAY CANONICAL REGRESSION")
print(f"Rows compared: {len(result_250)}")
print(f"Maximum quantile difference: {quantile_difference.max():.3e}")
print(f"Maximum VaR difference: {var_difference_250.max():.3e}")

Window 125
Rows: 1137
Observations: 125 to 125
Violations: 73
Minimum VaR: 1.239389%
Maximum VaR: 4.955767%
Maximum sign convention difference: 0.000e+00



Window 250
Rows: 1137
Observations: 250 to 250
Violations: 64
Minimum VaR: 1.708839%
Maximum VaR: 4.378557%
Maximum sign convention difference: 0.000e+00



Window 500
Rows: 1137
Observations: 500 to 500
Violations: 68
Minimum VaR: 1.949615%
Maximum VaR: 3.101157%
Maximum sign convention difference: 0.000e+00

250-DAY CANONICAL REGRESSION
Rows compared: 1137
Maximum quantile difference: 1.596e-16
Maximum VaR difference: 1.596e-16


### Sensitivity metrics

Evaluate all three rolling windows on the same 1,137 target dates using violation rate, Pinball Loss and VaR level metrics. This comparison is descriptive and does not change the canonical 250-day baseline.

In [10]:
sensitivity_summary_rows = []

for window in SENSITIVITY_WINDOWS:
    result = sensitivity_results[window]
    actual = result["actual_return"].to_numpy(dtype="float64")
    quantile = result["quantile_return"].to_numpy(dtype="float64")
    var = result["var"].to_numpy(dtype="float64")
    error = actual - quantile
    pinball = np.maximum(ALPHA * error, (ALPHA - 1.0) * error)

    assert np.isfinite(pinball).all()
    assert (pinball >= 0.0).all()

    sensitivity_summary_rows.append({
        "window": window, "forecasts": len(result), "violations": int(result["violation"].sum()),
        "violation_rate": float(result["violation"].mean()), "pinball_loss": float(pinball.mean()),
        "average_var": float(var.mean()), "minimum_var": float(var.min()), "maximum_var": float(var.max()),
    })

sensitivity_summary = pd.DataFrame(sensitivity_summary_rows)
baseline_sensitivity = sensitivity_summary.loc[sensitivity_summary["window"].eq(BASELINE_WINDOW)].iloc[0]
sensitivity_summary["rate_diff_pp_vs_5pct"] = (sensitivity_summary["violation_rate"] - ALPHA) * 100.0
sensitivity_summary["pinball_diff_vs_250"] = sensitivity_summary["pinball_loss"] - baseline_sensitivity["pinball_loss"]
sensitivity_summary["average_var_diff_pp_vs_250"] = (sensitivity_summary["average_var"] - baseline_sensitivity["average_var"]) * 100.0

assert sensitivity_summary["forecasts"].eq(1137).all()
assert sensitivity_summary["violations"].tolist() == [73, 64, 68]

display(sensitivity_summary)

print("Interpretation:")
print("125-day window has the lowest Pinball Loss but the highest violation rate.")
print("250-day window has the violation rate closest to the nominal 5% level among the three windows.")
print("500-day window has an intermediate violation rate but the highest Pinball Loss.")
print("The metrics indicate a trade-off rather than unanimous dominance by one window.")
print("The 250-day baseline remains unchanged because this sensitivity check is descriptive rather than a tuning procedure.")

,window,forecasts,violations,violation_rate,pinball_loss,average_var,minimum_var,maximum_var,rate_diff_pp_vs_5pct,pinball_diff_vs_250,average_var_diff_pp_vs_250
0,125,1137,73,0.064204,0.002050,0.025103,0.012394,0.049558,1.420405,-0.000008,-0.09038
1,250,1137,64,0.056288,0.002058,0.026007,0.017088,0.043786,0.628848,0.000000,0.00000
2,500,1137,68,0.059807,0.002079,0.025713,0.019496,0.031012,0.980651,0.000021,-0.02938


Interpretation:
125-day window has the lowest Pinball Loss but the highest violation rate.
250-day window has the violation rate closest to the nominal 5% level among the three windows.
500-day window has an intermediate violation rate but the highest Pinball Loss.
The metrics indicate a trade-off rather than unanimous dominance by one window.
The 250-day baseline remains unchanged because this sensitivity check is descriptive rather than a tuning procedure.


## Canonical baseline summary

Recompute the official Historical Simulation baseline metrics directly from the canonical row-level backtest. The interpretation is descriptive because no formal coverage hypothesis test is performed in this review.

In [11]:
actual = backtest["target_return"].to_numpy(dtype="float64")
quantile = backtest["quantile_return"].to_numpy(dtype="float64")
var = backtest["historical_var"].to_numpy(dtype="float64")
violations = actual < quantile
error = actual - quantile
pinball = np.maximum(ALPHA * error, (ALPHA - 1.0) * error)

forecast_count = len(backtest)
violation_count = int(violations.sum())
violation_rate = float(violations.mean())
expected_violation_rate = 1.0 - CONFIDENCE_LEVEL
expected_violation_count = forecast_count * expected_violation_rate
rate_difference_pp = (violation_rate - expected_violation_rate) * 100.0
pinball_loss = float(pinball.mean())
average_var = float(var.mean())

baseline_summary = pd.Series({
    "confidence_level": CONFIDENCE_LEVEL, "alpha": ALPHA, "forecasts": forecast_count,
    "violations": violation_count, "violation_rate": violation_rate,
    "expected_violation_rate": expected_violation_rate, "expected_violations": expected_violation_count,
    "rate_difference_pp": rate_difference_pp, "pinball_loss": pinball_loss,
    "average_var": average_var, "minimum_var": float(var.min()), "maximum_var": float(var.max()),
})

assert forecast_count == 1387
assert violation_count == 75
assert backtest["violation"].eq(violations).all()
assert np.isfinite(pinball).all()
assert (pinball >= 0.0).all()

display(baseline_summary.to_frame("value"))

interpretation = (
    f"Across {forecast_count:,} one-day forecasts, Historical Simulation produced {violation_count} VaR violations, "
    f"corresponding to a violation rate of {violation_rate:.6%} versus the nominal {expected_violation_rate:.0%} rate. "
    f"The observed rate is {rate_difference_pp:.6f} percentage points above the target, indicating mild undercoverage "
    f"or a small tendency to underestimate one-day lower-tail risk. The Pinball Loss is {pinball_loss:.12f} and "
    f"the Average VaR is {average_var:.6%}. These findings are a descriptive calibration assessment. "
    "No formal coverage hypothesis test is performed here, so this review does not establish statistical significance "
    "and does not interpret the result as a rejection or failure of VaR coverage."
)

print("REPORT-READY INTERPRETATION")
print(interpretation)

,value
confidence_level,0.950000
alpha,0.050000
forecasts,1387.000000
violations,75.000000
violation_rate,0.054074
expected_violation_rate,0.050000
expected_violations,69.350000
rate_difference_pp,0.407354
pinball_loss,0.002057
average_var,0.026231


REPORT-READY INTERPRETATION
Across 1,387 one-day forecasts, Historical Simulation produced 75 VaR violations, corresponding to a violation rate of 5.407354% versus the nominal 5% rate. The observed rate is 0.407354 percentage points above the target, indicating mild undercoverage or a small tendency to underestimate one-day lower-tail risk. The Pinball Loss is 0.002057488935 and the Average VaR is 2.623147%. These findings are a descriptive calibration assessment. No formal coverage hypothesis test is performed here, so this review does not establish statistical significance and does not interpret the result as a rejection or failure of VaR coverage.
